In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
import ipywidgets as widgets
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [2]:
"""
Go to the Earth Engine Code Editor. https://code.earthengine.google.com/
Click the Assets tab on the left.
Click New -> Shapefile.
Select your .shp, .shx, .dbf, and .prj files from your data_for_adm03_bangladesh folder.
Name it (e.g., bd_upazilas_level3).
Click Upload. (Wait a few minutes for the task to finish in the "Tasks" tab).
"""

upazilas = ee.FeatureCollection(f"projects/gen-lang-client-0291086733/assets/data_for_adm03_bangladesh")
print("Size:", upazilas.size().getInfo())

Size: 544


In [3]:
python_list_ee = upazilas.aggregate_array('ADM2_EN').zip(upazilas.aggregate_array('ADM3_EN'))
for i in range(5):
    print(python_list_ee.get(i).getInfo())

['Barguna', 'Amtali']
['Barguna', 'Bamna']
['Barguna', 'Barguna Sadar']
['Barguna', 'Betagi']
['Barguna', 'Patharghata']


In [4]:
m = geemap.Map()
m.setCenter(90.35, 23.68, 7)
m.set_options('HYBRID')

upazilas_with_random = upazilas.randomColumn('random_id', 1)
colorful_map = upazilas_with_random.reduceToImage(
    properties=['random_id'],
    reducer=ee.Reducer.first()
)

random_vis = {
    'min': 0, 
    'max': 1, 
    'palette': ['FF0000', '00FF00', '0000FF', 'FFFF00', '00FFFF', 'FF00FF', 'FFA500'] 
}

m.addLayer(colorful_map, random_vis, 'upazilas')
borders = ee.Image().paint(upazilas, 0, 1)
m.addLayer(borders, {'palette': 'black'}, 'borders')
m

Map(center=[23.68, 90.35], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [5]:
folder_path = 'data_for_adm03_bangladesh'

shp_files = [f for f in os.listdir(folder_path) if f.endswith('.shp')]

if not shp_files:
    raise FileNotFoundError(f"didn't found any shp files in {folder_path}")

shp_path = os.path.join(folder_path, shp_files[0])
print(f"this is the file {shp_path}")

this is the file data_for_adm03_bangladesh\bgd_admbnda_adm3_bbs_20180410.shp


In [6]:
import geopandas as gpd

#read and covert to geojson
gdf = gpd.read_file(shp_path)
print(f"Loaded {len(gdf)} features")
geojson = gdf.__geo_interface__



Loaded 544 features


In [7]:
python_list = list(zip(gdf['ADM2_EN'], gdf['ADM3_EN']))
print(python_list[:5])

[('Jessore', 'Abhaynagar'), ('Dhaka', 'Adabor'), ('Bogra', 'Adamdighi'), ('Lalmonirhat', 'Aditmari'), ('Barisal', 'Agailjhara')]


In [8]:
print("total upzillas: ", len(python_list))

total upzillas:  544


In [9]:
import numpy as np
np.random.seed(1)
gdf['random_id'] = np.random.rand(len(gdf))
def get_color(feature):
    value = feature['properties']['random_id']
    palette = ['#FF0000', '#00FF00', '#0000FF', '#FFFF00', '#00FFFF', '#FF00FF', '#FFA500']
    idx = int(value * len(palette))
    idx = min(idx, len(palette) - 1)
    return {
        'fillColor': palette[idx],
        'color': 'black', #this is border color
        'weight': 1,     #this is border weight
        'fillOpacity': 0.7
    }

m = geemap.Map()
m.setCenter(90.35, 23.68, 7)
m.add_gdf(
    gdf, 
    layer_name='Colorful Upazilas', 
    style_callback=get_color,
    info_mode='on_click' #for the inspactor map
)

m


Map(center=[23.68, 90.35], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…